In [9]:
import pandas as pd
import numpy as np
import lightgbm as lgb

RAW = '../data/raw/'
train = pd.read_csv(RAW + 'train.csv')
test  = pd.read_csv(RAW + 'test.csv')
train['tarih'] = pd.to_datetime(train['tarih'])
test['tarih']  = pd.to_datetime(test['tarih'])

def rmsle(gercek, tahmin):
    return np.sqrt(np.mean((np.log1p(tahmin) - np.log1p(gercek))**2))

print('train', train.shape, '| test', test.shape)

train (1226237, 5) | test (714688, 5)


EDA bulgusu: 444 satır fiziksel olarak imkansız (guc x 24 saat) teorik imkansız,
çoğunluğu sadece 4 trafoda, toplam 23 trafoda yoğun. Sorun lokalize olduğu icin global clipping
yerine trafo bazlı medyan ile düzelttik

In [10]:
train['max_teorik']  = train['guc'] * 24
train['asiri_oran']  = train['tuketim'] / train['max_teorik']
train['tuketim_duzeltilmis'] = train['tuketim']

problemli = train.loc[train['asiri_oran'] > 2, 'tanim'].unique()
for t in problemli:
    medyan = train.loc[(train['tanim']==t) & (train['asiri_oran']<=2), 'tuketim'].median()
    train.loc[(train['tanim']==t) & (train['asiri_oran']>2), 'tuketim_duzeltilmis'] = medyan

print('Düzeltilen satır:', (train['tuketim'] != train['tuketim_duzeltilmis']).sum(), '| problemli trafo:', len(problemli))

Düzeltilen satır: 444 | problemli trafo: 23


In [17]:
TATILLER = pd.to_datetime([
    # 2025
    '2025-01-01','2025-03-29','2025-03-30','2025-03-31','2025-04-01','2025-04-23',
    '2025-05-01','2025-05-19','2025-06-05','2025-06-06','2025-06-07','2025-06-08','2025-06-09',
    '2025-08-30','2025-10-29',
    # 2026 (test dönemi dahil)
    '2026-01-01','2026-03-19','2026-03-20','2026-03-21','2026-04-23','2026-05-01','2026-05-19',
    '2026-05-26','2026-05-27','2026-05-28','2026-05-29','2026-05-30','2026-08-30','2026-10-29',
]).date

GUC_BINS = [0, 100, 250, 500, 1000, 2000, 5000, np.inf]

def temel_ozellikler(df):
    df = df.copy()
    lok = df['lokasyon'].str.split('>', expand=True)
    df['il']    = lok[0]
    df['bolge'] = lok[1]
    df['ilce']  = (lok[2] if lok.shape[1] > 2 else pd.Series(np.nan, index=df.index)).fillna('YOK')

    df['ay']            = df['tarih'].dt.month
    df['gun']           = df['tarih'].dt.day
    df['haftanin_gunu'] = df['tarih'].dt.dayofweek
    df['hafta_sonu']    = (df['haftanin_gunu'] >= 5).astype(int)
    df['yilin_gunu']    = df['tarih'].dt.dayofyear
    df['tatil']         = df['tarih'].dt.date.isin(TATILLER).astype(int)

    df['guc_bucket']     = pd.cut(df['guc'], bins=GUC_BINS)
    df['guc_bucket_str'] = df['guc_bucket'].astype(str)
    return df

train = temel_ozellikler(train)
test  = temel_ozellikler(test)

KATEGORIK = ['il','bolge','ilce','guc_bucket_str']
print('train ->', train.shape, '| test ->', test.shape)
print('tatil işaretli satır -> train:', train['tatil'].sum(), '| test ->', test['tatil'].sum())

train -> (1226237, 19) | test -> (714688, 16)
tatil işaretli satır -> train: 48245 | test -> 48256


In [12]:
#trafo profili log uzayda

def trafo_profili_olustur(gecmis, origin_tarihi):
    g = gecmis[gecmis['tarih'] < origin_tarihi].copy()
    g['log_t'] = np.log1p(g['tuketim_duzeltilmis'])

    profil = g.groupby('tanim').agg(
        trafo_ortalama   = ('tuketim_duzeltilmis','mean'),
        trafo_medyan     = ('tuketim_duzeltilmis','median'),
        trafo_gun_sayisi = ('tuketim_duzeltilmis','count'),
        log_ortalama     = ('log_t','mean'),
        log_medyan       = ('log_t','median'),
        log_std          = ('log_t','std'),
        sifir_orani      = ('tuketim_duzeltilmis', lambda x: (x == 0).mean()),
    ).reset_index()

    for pencere in [30, 90]:
        son  = g[g['tarih'] >= origin_tarihi - pd.Timedelta(days=pencere)]
        ozet = son.groupby('tanim').agg(**{
            f'son{pencere}_log_ort': ('log_t','mean'),
            f'son{pencere}_sifir'  : ('tuketim_duzeltilmis', lambda x: (x == 0).mean()),
        }).reset_index()
        profil = profil.merge(ozet, on='tanim', how='left')

    profil['log_trend'] = profil['son30_log_ort'] - profil['son90_log_ort']
    return profil


def trafo_gun_profili(gecmis, origin_tarihi):
    g = gecmis[gecmis['tarih'] < origin_tarihi][['tanim','haftanin_gunu','tuketim_duzeltilmis']].copy()
    ort = g.groupby('tanim')['tuketim_duzeltilmis'].mean().rename('_ort')
    g = g.merge(ort, on='tanim', how='left')
    g['oran'] = g['tuketim_duzeltilmis'] / g['_ort'].replace(0, np.nan)
    return g.groupby(['tanim','haftanin_gunu'])['oran'].mean().rename('trafo_gun_orani').reset_index()

veri_seti_kur

Tek fonksiyon, 3 yerde aynı şekilde kullanılır (egitim / validation / test) —> sadece
origin_tarihi değişir. Bu, egitim ve tahmin koşullarının birebir aynı olmasını garantiler.

- Grup istatistiği shrinkage'li (k=30): küçükk gruplarda ham ortalama gürültülü
  (örnek: 97 gruptan 3'ü <50 satır; bir grubun ham ortalamasi 0'dı).
- iki kademeli yedek: il+bolge+guc -> il+guc -> genel ortalama.
- ufuk_gun: modelin kaç gun ileriyi tahmin ettiğini bilmesi için (hata ufukla büyüyor).

In [13]:
def veri_seti_kur(gecmis, hedef_satirlar, origin_tarihi, k=30):
    g = gecmis[gecmis['tarih'] < origin_tarihi].copy()
    g['log_t']  = np.log1p(g['tuketim_duzeltilmis'])
    genel_log   = g['log_t'].mean()

    grup = g.groupby(['il','bolge','guc_bucket'], observed=True)['log_t'].agg(
        ['sum','count']).reset_index()
    grup['grup_log_ort'] = (grup['sum'] + k * genel_log) / (grup['count'] + k)

    il_guc = (g.groupby(['il','guc_bucket'], observed=True)['log_t'].mean().rename('ilguc_log_ort').reset_index())

    profil   = trafo_profili_olustur(gecmis, origin_tarihi)
    gun_prof = trafo_gun_profili(gecmis, origin_tarihi)

    df = hedef_satirlar.drop(columns=['grup_log_ort','ilguc_log_ort'], errors='ignore').copy()
    df = df.merge(grup[['il','bolge','guc_bucket','grup_log_ort']], on=['il','bolge','guc_bucket'], how='left')
    df = df.merge(il_guc, on=['il','guc_bucket'], how='left')
    df['grup_log_ort'] = df['grup_log_ort'].fillna(df['ilguc_log_ort']).fillna(genel_log)

    df = df.merge(profil,   on='tanim', how='left')
    df = df.merge(gun_prof, on=['tanim','haftanin_gunu'], how='left')

    df['ufuk_gun'] = (df['tarih'] - origin_tarihi).dt.days
    return df


DIRECT_FEATURES = [
    'guc','ay','gun','haftanin_gunu','hafta_sonu','yilin_gunu','tatil',
    'il','bolge','ilce','guc_bucket_str','grup_log_ort',
    'trafo_ortalama','trafo_medyan','trafo_gun_sayisi',
    'log_ortalama','log_medyan','log_std','sifir_orani',
    'son30_log_ort','son90_log_ort','son30_sifir','son90_sifir',
    'log_trend','trafo_gun_orani','ufuk_gun',
]
print('Özellik sayısı:', len(DIRECT_FEATURES))

Özellik sayısı: 26


In [14]:

EGITIM_ORIGIN = pd.Timestamp('2025-08-01')
VAL_ORIGIN    = pd.Timestamp('2025-12-01')

egitim_seti = veri_seti_kur(train, train[(train['tarih'] >= EGITIM_ORIGIN) & (train['tarih'] <= '2025-11-30')], EGITIM_ORIGIN)
val_seti    = veri_seti_kur(train, train[ train['tarih'] >= VAL_ORIGIN], VAL_ORIGIN)

for df in [egitim_seti, val_seti]:
    for c in KATEGORIK:
        df[c] = df[c].astype('category')
    df['hedef_log'] = np.log1p(df['tuketim_duzeltilmis'])

print('egitim', egitim_seti.shape, '| validation', val_seti.shape)
print('validation ufku:', val_seti['ufuk_gun'].max(), 'gun')

egitim (319732, 36) | validation (444076, 36)
validation ufku: 120 gun


In [15]:
direct_model = lgb.LGBMRegressor(
    n_estimators=3000, learning_rate=0.05, num_leaves=63, random_state=42
)
direct_model.fit(
    egitim_seti[DIRECT_FEATURES], egitim_seti['hedef_log'],
    eval_set=[(val_seti[DIRECT_FEATURES], val_seti['hedef_log'])],
    eval_metric='rmse',
    categorical_feature=KATEGORIK,
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
)

val_seti['tahmin'] = np.clip(np.expm1(direct_model.predict(val_seti[DIRECT_FEATURES])), 0, None)
print('Override oncesi RMSLE :', rmsle(val_seti['tuketim_duzeltilmis'], val_seti['tahmin']))

val_seti.loc[val_seti['sifir_orani'] >= 0.9, 'tahmin'] = 0.0
print('Override sonrasi RMSLE:', rmsle(val_seti['tuketim_duzeltilmis'], val_seti['tahmin']))
print('best_iteration        :', direct_model.best_iteration_)

/Users/beratmertkayacan/Desktop/GridUpDatathon/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010764 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3136
[LightGBM] [Info] Number of data points in the train set: 319732, number of used features: 26
[LightGBM] [Info] Start training from score 6.549641
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[80]	valid_0's rmse: 1.02377	valid_0's l2: 1.0481
Override oncesi RMSLE : 1.0237664720557413
Override sonrasi RMSLE: 1.0215138710698266
best_iteration        : 80


In [16]:
onem = pd.DataFrame({'ozellik': DIRECT_FEATURES, 'onem': direct_model.feature_importances_})
print(onem.sort_values('onem', ascending=False).to_string(index=False))

         ozellik  onem
      yilin_gunu   718
            ilce   568
             guc   480
   son30_log_ort   411
           bolge   386
trafo_gun_sayisi   309
  trafo_ortalama   297
         log_std   266
       log_trend   232
    grup_log_ort   228
    log_ortalama   194
 trafo_gun_orani   180
   son90_log_ort   167
    trafo_medyan   149
     son30_sifir   115
  guc_bucket_str    98
     sifir_orani    70
              ay    21
     son90_sifir    21
      log_medyan    19
             gun    18
              il    11
   haftanin_gunu     2
           tatil     0
      hafta_sonu     0
        ufuk_gun     0
